# Download the Hearing Transcripts and MODS Records

| Item | Description |
|---|---|
| Goal | Fetch two files for every package in the manifest: the MODS metadata record and the HTML transcript. Record an outcome for every package, so the number missing is a subtraction against the manifest and not a guess from what is on disk. |
| Input | `Raw/manifest/packages_all.parquet` from D1.1. The GovInfo API, `/packages/{packageId}/mods` and `/packages/{packageId}/htm`. The API key in `.env` in the D folder. |
| Output | In the D1 Raw folder: `<year>/mods/<packageId>.xml`, one per package; `<year>/html/<packageId>.htm`, one per package that has an HTML rendition; `status/fetch_<year>.jsonl`, one line per package and file kind; `status/fetch_summary.parquet`, all status lines in one table. |
| Run | Last run 2026-08-20, two files for each of 36,378 packages over about two hours. The notebook kept no cell outputs. Not rerun: the files are on disk, and a repeat would load a shared public API for nothing. Step 8 reads the status log of that run; it was executed on 2026-10-31 against the copied files and downloaded nothing. |

## Step 1. Import packages

`requests` fetches the files. `json` writes the status lines.

In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

## Step 2. Define file locations

Files go to `Raw/<year>/mods` and `Raw/<year>/html`, by the issue year the manifest uses. One year can then be collected again on its own. `KINDS` maps the API's file-kind token to the folder and the extension on disk. The API serves the transcript at `/htm` and the metadata at `/mods`, and the file names say what the files are.

In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("D - Data Collection and Processing/D1 - GovInfo Congressional Hearings")
NOTEBOOK_NAME = "D1.2 - GovInfo - Download Hearings"
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
MANIFEST_FOLDER = RAW_FOLDER / "manifest"  # written by D1.1
STATUS_FOLDER = RAW_FOLDER / "status"  # one status log per year
ENV_FILE = PROJECT_FOLDER / "D - Data Collection and Processing" / ".env"  # holds GOVINFO_API_KEY

# API file-kind token -> (folder under Raw/<year>, extension on disk)
KINDS = {"mods": ("mods", ".xml"), "htm": ("html", ".htm")}

STATUS_FOLDER.mkdir(parents=True, exist_ok=True)

## Step 3. Load the manifest and read the API key

The manifest is the work list. Every row is attempted, and a skip is recorded as a skip.

In [ ]:
manifest = pd.read_parquet(MANIFEST_FOLDER / "packages_all.parquet")

API_KEY = os.environ.get("GOVINFO_API_KEY")
if not API_KEY and ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():  # one KEY=value per line
        name, _, value = line.strip().partition("=")
        if name == "GOVINFO_API_KEY" and value:
            API_KEY = value.strip().strip('"').strip("'")
assert API_KEY, f"GOVINFO_API_KEY is not set and {ENV_FILE} does not supply it; register a free key at https://api.data.gov/signup/"

print(f"{len(manifest):,} packages to fetch, {len(KINDS)} files each = {len(manifest) * len(KINDS):,} requests at most")
print(f"years {manifest['query_year'].min()}-{manifest['query_year'].max()}")

## Step 4. Define one API request

The same request as in D1.1, with one addition. GovInfo answers a request for a rendition a package does not have with a 400 or 404. The body then says the resource does not exist. That answer is a fact about the package, not a failure, so the request reports a missing file instead of an error.

In [ ]:
BASE_URL = "https://api.govinfo.gov"
SECONDS_BETWEEN_REQUESTS = 0.1  # 10 requests per second, a quarter of the published limit for a registered key
MAX_ATTEMPTS = 5  # per request; the wait doubles after each failure
TRANSIENT_STATUS = {408, 425, 429, 500, 502, 503, 504}  # retried; any other error code stops the notebook
MISSING_STATUS = {400, 404}  # with a body saying the resource does not exist: the package has no such file

session = requests.Session()
request_log = {"requests": 0, "retries": 0, "last_request": 0.0}


def govinfo_get(url):
    """One GET. Retries a transient failure, reports a missing rendition, raises on anything else."""
    last_error = "no attempt made"
    for attempt in range(MAX_ATTEMPTS):
        wait = SECONDS_BETWEEN_REQUESTS - (time.monotonic() - request_log["last_request"])
        if wait > 0:
            time.sleep(wait)
        request_log["last_request"] = time.monotonic()
        request_log["requests"] += 1
        try:
            response = session.get(url, headers={"X-Api-Key": API_KEY}, timeout=120)
        except requests.RequestException as exc:  # connection reset, DNS failure, timeout
            last_error = f"{type(exc).__name__}: {exc}"
        else:
            if response.status_code in MISSING_STATUS and "does not exist" in response.text.lower():
                raise FileNotFoundError(f"HTTP {response.status_code}, no such rendition: {url}")
            if response.status_code < 400:
                return response
            if response.status_code not in TRANSIENT_STATUS:
                raise RuntimeError(f"HTTP {response.status_code} for {url}: {response.text[:200]}")
            last_error = f"HTTP {response.status_code}"
            retry_after = response.headers.get("Retry-After")
            if retry_after and retry_after.isdigit():
                time.sleep(int(retry_after))
                request_log["retries"] += 1
                continue
        request_log["retries"] += 1
        time.sleep(2.0 ** attempt)
    raise RuntimeError(f"{MAX_ATTEMPTS} attempts failed for {url}: {last_error}")

## Step 5. Define one package, one recorded outcome

Every path through `fetch_one` returns a status record with one of four statuses: `ok`, `skipped`, `missing`, or `error`. `missing` means GovInfo has no such rendition. `error` means a transient failure outlived the retries. A file is written under a temporary name and then renamed, so an interrupted write cannot leave a truncated file.

In [ ]:
def fetch_one(package_id, year, kind):
    """Fetch one rendition of one package and return what happened: ok, skipped, missing, or error."""
    subdir, extension = KINDS[kind]
    folder = RAW_FOLDER / str(year) / subdir
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / f"{package_id}{extension}"
    record = {
        "packageId": package_id,
        "year": year,
        "kind": kind,
        "path": str(target),
        "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

    if target.exists() and target.stat().st_size > 0:
        return {**record, "status": "skipped", "bytes": target.stat().st_size}

    try:
        payload = govinfo_get(f"{BASE_URL}/packages/{package_id}/{kind}").content
    except FileNotFoundError:
        return {**record, "status": "missing", "bytes": 0}
    except RuntimeError as exc:
        return {**record, "status": "error", "bytes": 0, "message": str(exc)[:300]}

    partial = target.with_suffix(target.suffix + ".partial")
    partial.write_bytes(payload)
    partial.replace(target)
    return {**record, "status": "ok", "bytes": len(payload)}

## Step 6. Walk the manifest, year by year

Status lines are appended to the year's log as they happen, so an interrupted run resumes instead of restarting. The set of packages already done is read from the log, not from the file system. One line is printed per year; a progress bar would leave thousands of lines in the saved notebook.

In [ ]:
counts = {"ok": 0, "skipped": 0, "missing": 0, "error": 0}
started = datetime.now(timezone.utc)

for year, group in manifest.groupby("query_year", sort=True):
    status_path = STATUS_FOLDER / f"fetch_{year}.jsonl"
    done = set()
    if status_path.exists():
        for line in status_path.read_text().splitlines():
            row = json.loads(line)
            if row["status"] in {"ok", "skipped", "missing"}:
                done.add((row["packageId"], row["kind"]))

    todo = [(package_id, kind) for package_id in group["packageId"] for kind in KINDS if (package_id, kind) not in done]
    if not todo:
        print(f"{year}  {len(group):>6,} packages | already complete")
        continue

    year_counts = {"ok": 0, "skipped": 0, "missing": 0, "error": 0}
    with status_path.open("a") as log:
        for package_id, kind in todo:
            record = fetch_one(package_id, int(year), kind)
            log.write(json.dumps(record) + "\n")
            log.flush()
            year_counts[record["status"]] += 1
            counts[record["status"]] += 1

    elapsed = (datetime.now(timezone.utc) - started).total_seconds()
    print(f"{year}  {len(group):>6,} packages | " + " ".join(f"{k}={v:,}" for k, v in year_counts.items() if v)
          + f" | {elapsed / 60:.0f} min elapsed")

print()
print("totals:", " ".join(f"{k}={v:,}" for k, v in counts.items()))
print(f"{request_log['requests']:,} API requests, {request_log['retries']:,} retries")

## Step 7. Consolidate the status log

Every status line from every year in one table. This is what the check below and every later stage read before assuming a transcript is on disk.

In [ ]:
status_frames = [pd.read_json(path, lines=True) for path in sorted(STATUS_FOLDER.glob("fetch_*.jsonl"))]
status = pd.concat(status_frames, ignore_index=True)
status.to_parquet(STATUS_FOLDER / "fetch_summary.parquet", index=False)

pivot = status.pivot_table(index="year", columns="status", values="packageId", aggfunc="count", fill_value=0)
print(pivot.to_string())
print()
print(f"total bytes written: {status.loc[status['status'] == 'ok', 'bytes'].sum() / 1e9:.1f} GB")

## Step 8. Check the download against the manifest

The manifest is the denominator for all five checks. This cell reads the saved status log, so it can run at any time without a download. A package and kind attempted more than once has one line per attempt, and the last attempt counts.

- No `error` rows remain.
- Every manifest package has a MODS record.
- Every package has one outcome per file kind.
- The files on disk match the `ok` and `skipped` rows exactly.
- The `missing` rate is reported, not asserted, because some older hearings have no HTML rendition.

In [1]:
status = pd.read_parquet(STATUS_FOLDER / "fetch_summary.parquet")
manifest = pd.read_parquet(MANIFEST_FOLDER / "packages_all.parquet")

# One line per attempt: a package retried after an error appears twice, and its last attempt is the outcome.
attempts = len(status)
status = status.sort_values("fetched_at").drop_duplicates(["packageId", "kind"], keep="last")
print(f"{attempts:,} attempts in the log, {len(status):,} package-kind outcomes ({attempts - len(status):,} retried)\n")

errors = status.loc[status["status"] == "error"]
assert len(errors) == 0, f"{len(errors)} packages failed after retries; run the notebook again to resume"
print("1) no unresolved fetch errors")

have_mods = set(status.loc[(status["kind"] == "mods") & status["status"].isin(["ok", "skipped"]), "packageId"])
missing_mods = set(manifest["packageId"]) - have_mods
assert not missing_mods, f"{len(missing_mods)} packages have no MODS record, e.g. {sorted(missing_mods)[:5]}"
print(f"2) all {len(have_mods):,} packages have a MODS record")

# The log may hold lines for other file kinds or for packages outside this manifest; they are reported, not counted.
core = status.loc[status["kind"].isin(KINDS) & status["packageId"].isin(manifest["packageId"])]
extra = status.loc[~status.index.isin(core.index)]
expected = len(manifest) * len(KINDS)
assert len(core) == expected, f"expected {expected:,} outcomes, log holds {len(core):,}"
print(f"3) {len(core):,} outcomes recorded, one per package per file kind"
      + (f"; {len(extra):,} further lines in the log, by kind {extra['kind'].value_counts().to_dict()}" if len(extra) else ""))
status = core

# Compare by <year>/<folder>/<file>: the log holds the absolute paths of the machine that ran it, and one
# listing per folder is far cheaper than one stat() per row on a shared file system.
present = set()
for subdir, _ in KINDS.values():
    for year_folder in sorted(RAW_FOLDER.glob(f"*/{subdir}")):
        present.update(f"{year_folder.parent.name}/{subdir}/{p.name}" for p in year_folder.iterdir())
claimed = set("/".join(Path(p).parts[-3:]) for p in status.loc[status["status"].isin(["ok", "skipped"]), "path"])
absent = claimed - present
orphan = present - claimed
assert not absent, f"log claims {len(absent):,} files that are not on disk, e.g. {sorted(absent)[:3]}"
assert not orphan, f"{len(orphan):,} files on disk have no status row, e.g. {sorted(orphan)[:3]}"
print(f"4) {len(claimed):,} files on disk match the status log exactly")

missing = status.loc[status["status"] == "missing"]
print(f"5) {len(missing):,} renditions absent from GovInfo ({len(missing) / len(status):.2%} of attempts)")
if len(missing):
    print(missing.groupby(["year", "kind"]).size().to_string())

73,221 attempts in the log, 73,221 package-kind outcomes (0 retried)

1) no unresolved fetch errors
2) all 36,378 packages have a MODS record
3) 72,756 outcomes recorded, one per package per file kind; 465 further lines in the log, by kind {'pdf': 465}
4) 71,881 files on disk match the status log exactly
5) 875 renditions absent from GovInfo (1.20% of attempts)
year  kind
2000  htm       8
2001  htm      43
2002  htm      67
2003  htm      76
2004  htm      37
2005  htm      61
2006  htm      69
2007  htm     121
2008  htm     125
2009  htm      88
2010  htm     100
2011  htm      49
2012  htm      21
2013  htm       4
2017  htm       1
2019  htm       2
2022  htm       1
2025  htm       2
